# 대화 기록 — 주피터 따라 하기
이 노트북은 **완성된 코드**입니다. 위에서부터 한 셀씩 실행하며, **같은 대화방**과 **새 대화방**에 같은 후속 질문을 보냈을 때 답과 실제 저장된 기록이 어떻게 다른지 확인합니다.

- 모든 셀은 ▶ 제공 코드입니다. 빈칸이나 채점 셀은 없습니다.
- 셀을 두 번 실행하면 같은 대화방에 메시지가 한 번 더 쌓입니다. 처음부터 다시 보려면 위에서부터 다시 실행하세요.
- 직접 구현하는 과제는 별도 런박스에서 합니다. 같은 폴더의 `바이브코딩_정정과대화방/실습_정정과대화방_바이브코딩.html`을 여세요.

이 노트북은 **독립 Runbox용**입니다. 다른 노트북을 열지 않아도 필요한 코드가 모두 들어 있습니다.

### 0. 실습 환경 확인
Python 버전이 **3.12.x**(3.12로 시작)로 나오면 정상입니다.

In [1]:
%pip install langchain langchain-core langchain-openai langgraph

  Using cached langgraph-1.2.24-py3-none-any.whl.metadata (4.9 kB)
  Using cached pydantic-2.14.4-py3-none-any.whl.metadata (143 kB)
  Using cached certifi-2026.11.20-py3-none-any.whl.metadata (2.5 kB)
  Using cached httpx-0.28.1-py3-none-any.whl.metadata (7.1 kB)
  Using cached jsonpatch-1.48-py3-none-any.whl.metadata (4.9 kB)
  Using cached langchain_protocol-0.0.22-py3-none-any.whl.metadata (2.4 kB)
  Using cached langsmith-0.16.2-py3-none-any.whl.metadata (40 kB)
  Using cached tenacity-9.2.1-py3-none-any.whl.metadata (1.4 kB)
  Using cached uuid_utils-1.2.0-cp310-abi3-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (6.4 kB)
  Using cached langgraph_checkpoint-4.3.2-py3-none-any.whl.metadata (6.7 kB)
  Using cached langgraph_prebuilt-1.1.3-py3-none-any.whl.metadata (6.9 kB)
  Using cached langgraph_sdk-0.4.14-py3-none-any.whl.metadata (5.1 kB)
  Using cached xxhash-4.1.0-cp312-cp312-manylinux2014_x86_64.manylinux_2_17_x86_64.manylinux_2_28_x86_64.whl.metadata (18 kB)
  Usin

Using cached openai-3.59.0-py3-none-any.whl (2.5 MB)
Using cached jiter-0.19.0-cp312-cp312-manylinux2014_x86_64.manylinux_2_17_x86_64.whl (338 kB)
Using cached orjson-3.13.0-cp312-cp312-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (130 kB)
Using cached ormsgpack-1.13.1-cp312-cp312-manylinux2014_x86_64.manylinux_2_17_x86_64.whl (273 kB)
Using cached requests-2.34.2-py3-none-any.whl (73 kB)
Using cached charset_normalizer-3.5.3-cp312-cp312-manylinux1_x86_64.manylinux_2_28_x86_64.manylinux_2_5_x86_64.whl (257 kB)
Using cached urllib3-2.8.0-py3-none-any.whl (135 kB)
Using cached requests_toolbelt-1.0.0-py2.py3-none-any.whl (54 kB)
Using cached tenacity-9.2.1-py3-none-any.whl (32 kB)
Using cached tiktoken-0.14.0-cp312-cp312-manylinux_2_28_x86_64.whl (1.2 MB)
Using cached truststore-0.10.5-py3-none-any.whl (19 kB)
Using cached typing_inspection-0.4.5-py3-none-any.whl (14 kB)
Using cached websockets-17.4-cp312-cp312-manylinux1_x86_64.manylinux_2_28_x86_64.manylinux_2_5_x86_64.whl (226 kB)
U

In [2]:
# ▶ 제공 코드 — 실습 환경(커널) 확인
import sys
from importlib.metadata import version

print("Python", sys.version.split()[0], "(기준 3.12.x)")
for name in ["langchain", "langchain-core", "langchain-openai", "langgraph"]:
    print(f"{name:18s}", version(name))

Python 3.12.9 (기준 3.12.x)
langchain          1.5.2
langchain-core     1.8.5
langchain-openai   1.8.5
langgraph          1.2.24


### 1. 설정 셀 — 모델 연결
안내받은 **수업용 API 키**(`API_KEY`)와 **채팅 모델 주소**(`BASE_URL`)를 따옴표 안에 직접 입력하세요. 모델은 `gpt-6-luna`로 정해져 있어 바꾸지 않습니다. 이 실습은 채팅 모델만 쓰므로 임베딩 설정은 없습니다.
- 두 값 중 하나라도 자리표시자로 남아 있으면 이 셀이 모델을 호출하기 전에 멈추고 무엇을 입력할지 알려 줍니다.
- `.env` 파일은 사용하지 않습니다.
- 이 셀은 AI 채팅창이나 AI 코딩 도우미에 **붙여넣지 마세요**. 노트북을 제출·공유하기 전에는 키를 다시 `"여기에_수업용_API_키_입력"`으로 되돌립니다.

In [3]:
# ▶ 제공 코드 — 설정 셀 (따옴표 안의 값만 바꾸세요)
API_KEY = ""
BASE_URL = ""   # 안내받은 채팅 모델 주소(https://...)
MODEL_NAME = "gpt-6-luna"                  # 수업에서 쓰는 채팅·도구 호출 모델(확정값, 바꾸지 않음)

missing = [name for name, value in (("API_KEY", API_KEY), ("BASE_URL", BASE_URL)) if value.startswith("여기에")]
if missing:
    raise ValueError(", ".join(missing) + "에 안내받은 값을 따옴표 안에 입력한 뒤 다시 실행하세요. (모델을 호출하기 전에 멈췄습니다)")

from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model=MODEL_NAME, api_key=API_KEY, base_url=BASE_URL)
print("모델 준비 완료:", MODEL_NAME)

모델 준비 완료: gpt-6-luna


### 2. 프롬프트 체인 기준 코드 — 체인은 기억하지 않는다
프롬프트 체인 실습에서 만든 인계문 체인입니다. 아래 두 번째 셀처럼 따로 보낸 두 질문은 서로 이어지지 않습니다.

In [4]:
# ▶ 제공 코드 — 프롬프트 체인 기준 코드 (가상 메모 · 인계 규칙 · handover_chain)
memo = """A라인 온도 센서 교체 완료.
14시 재가동 예정.
재가동 후 30분 동안 온도 로그 확인 필요."""

HANDOVER_RULES = """당신은 제조 현장의 교대 인계 도우미입니다.
메모를 정확히 세 줄로만 정리하세요.
1줄: '완료: '로 시작하고 끝난 작업을 씁니다.
2줄: '예정: '로 시작하고 앞으로 할 작업과 시각을 씁니다.
3줄: '확인: '로 시작하고 다음 근무자가 확인할 일을 씁니다.
메모에 없는 내용은 만들지 말고, 해당 내용이 없으면 '기재 없음'이라고 씁니다.
'예정'인 일을 '완료'로 바꾸지 마세요."""

from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

parser = StrOutputParser()
handover_prompt = ChatPromptTemplate.from_messages([
    ("system", HANDOVER_RULES),
    ("human", "{memo}"),
])
handover_chain = handover_prompt | llm | parser   # 프롬프트 → 모델 → 파서

print(handover_chain.invoke({"memo": memo}))

완료: A라인 온도 센서 교체
예정: 14시 재가동
확인: 재가동 후 30분 동안 온도 로그 확인


In [5]:
# ▶ 제공 코드 — 따로 보낸 두 질문은 서로 이어지지 않습니다
print(llm.invoke("A라인은 14시에 재가동할 예정이야.").content)
print("-" * 40)
print(llm.invoke("A라인 재가동은 몇 시야?").content)   # 앞 문장을 받지 못했습니다

알겠습니다. A라인은 14시에 재가동 예정으로 확인했습니다.
----------------------------------------
현재 대화에는 A라인 재가동 시간이 나와 있지 않아요. 관련 공지나 일정을 보내주시면 확인해 드릴게요.


### 3. 대화 기록 붙이기 — `create_agent` + `InMemorySaver`
- `checkpointer=InMemorySaver()` : 이 노트북이 실행되는 동안 **대화방별 기록**(메시지 목록)을 보관합니다.
- `config={"configurable": {"thread_id": "A-1"}}` : 이번 호출이 **어느 대화방**인지 알려 줍니다.
- 호출할 때는 **새 사용자 메시지 하나만** 보냅니다. 이전 메시지는 checkpointer가 같은 thread_id의 기록에서 꺼내 붙입니다.

아래 셀은 이후 계속 쓰는 함수 네 개(`new_chat_agent`, `config_for`, `ask`, `show_history`)를 만듭니다.

In [6]:
# ▶ 제공 코드 — 대화 기록 에이전트와 도우미 함수
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver

CHAT_RULES = """당신은 제조 현장의 교대 인계 도우미입니다.
이 대화에서 사용자가 알려준 내용만 근거로 답하세요.
같은 항목이 정정되면 가장 최근에 알려준 값을 따르세요.
대화에 없는 정보는 추측하지 말고 확인을 요청하세요."""

def new_chat_agent():
    """빈 기억 저장소(InMemorySaver)를 가진 대화 에이전트를 새로 만듭니다."""
    return create_agent(model=llm, tools=[], system_prompt=CHAT_RULES,
                        checkpointer=InMemorySaver())

def config_for(thread_id: str) -> dict:
    """대화방 이름(thread_id)을 실행 설정(config)으로 바꿉니다."""
    return {"configurable": {"thread_id": thread_id}}

def ask(thread_id: str, text: str) -> str:
    """새 사용자 메시지 하나만 보냅니다. 이전 대화는 checkpointer가 복원합니다."""
    result = chat_agent.invoke(
        {"messages": [{"role": "user", "content": text}]},
        config_for(thread_id),
    )
    answer = result["messages"][-1].content
    print(f"[{thread_id}] 사용자: {text}")
    print(f"[{thread_id}] 답변  : {answer}\n")
    return answer

def show_history(thread_id: str, agent=None):
    """해당 thread_id에 실제로 저장된 메시지를 순서대로 보여 줍니다."""
    agent = agent or chat_agent
    messages = agent.get_state(config_for(thread_id)).values.get("messages", [])
    print(f"--- [{thread_id}] 저장된 메시지 {len(messages)}개 ---")
    for i, m in enumerate(messages, 1):
        if m.type == "ai" and m.tool_calls:
            calls = ", ".join(f"{c['name']}({c['args']})" for c in m.tool_calls)
            print(f"{i:2d}. ai   → 도구 요청: {calls}")
        elif m.type == "tool":
            print(f"{i:2d}. tool → {m.name} 반환값: {m.content}")
        else:
            print(f"{i:2d}. {m.type:5s}: {m.content}")
    return messages

chat_agent = new_chat_agent()

print("chat_agent 준비 완료")

chat_agent 준비 완료


### 4. 같은 질문을 기존 대화방과 새 대화방에 보내기
`A-1`에는 먼저 정보를 알려 주고 질문합니다. `B-1`은 처음 열린 대화방에서 같은 질문을 받습니다.

In [7]:
# ▶ 제공 코드 — 같은 질문, 다른 대화방
ask("A-1", "A라인은 14시에 재가동할 예정이야.")
ask("A-1", "A라인 재가동은 몇 시야?")
ask("B-1", "A라인 재가동은 몇 시야?")

[A-1] 사용자: A라인은 14시에 재가동할 예정이야.
[A-1] 답변  : A라인은 14시에 재가동 예정으로 기록했습니다.

[A-1] 사용자: A라인 재가동은 몇 시야?
[A-1] 답변  : A라인 재가동은 14시 예정입니다.

[B-1] 사용자: A라인 재가동은 몇 시야?
[B-1] 답변  : 아직 A라인 재가동 시간은 전달받지 못했어요. 재가동 예정 시간을 알려주세요.



'아직 A라인 재가동 시간은 전달받지 못했어요. 재가동 예정 시간을 알려주세요.'

### 5. 응답만 보지 말고 실제 저장된 기록 확인하기
`get_state(config)`로 대화방에 저장된 메시지를 엽니다. 사용자 질문과 모델 답변을 각각 한 개로 셉니다. 위에서부터 한 번씩 실행했다면 **A-1에는 4개**, **B-1에는 2개**가 있습니다. B-1의 답에 우연히 `14시`가 나오더라도, 그 정보가 어느 대화방에 있었는지는 저장된 기록으로 판정합니다.

In [8]:
# ▶ 제공 코드 — 대화방별 실제 기록
show_history("A-1")
show_history("B-1")

--- [A-1] 저장된 메시지 4개 ---
 1. human: A라인은 14시에 재가동할 예정이야.
 2. ai   : A라인은 14시에 재가동 예정으로 기록했습니다.
 3. human: A라인 재가동은 몇 시야?
 4. ai   : A라인 재가동은 14시 예정입니다.
--- [B-1] 저장된 메시지 2개 ---
 1. human: A라인 재가동은 몇 시야?
 2. ai   : 아직 A라인 재가동 시간은 전달받지 못했어요. 재가동 예정 시간을 알려주세요.


[HumanMessage(content='A라인 재가동은 몇 시야?', additional_kwargs={}, response_metadata={}, id='7b645707-8ec3-4336-be89-b0e2b051dda6'),
 AIMessage(content='아직 A라인 재가동 시간은 전달받지 못했어요. 재가동 예정 시간을 알려주세요.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 53, 'prompt_tokens': 86, 'total_tokens': 139, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 20, 'rejected_prediction_tokens': 0, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': 0, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}, 'input_tokens': 86, 'output_tokens': 53, 'cached_prompt_tokens': 0, 'cache_write_tokens': 0}, 'model_provider': 'openai', 'model_name': 'gpt-6-luna', 'system_fingerprint': None, 'id': 'chatcmpl-EoN6a9gsc11h3XpETM4Lw2eTFkJfX', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None, 'normalized_finish_reason': 'stop'}, id='lc_run--01a21681-a304-7624-9391-347

### 6. 같은 대화방에서 정정하기
정정은 지난 메시지를 지우지 않습니다. 기록에는 14시와 15시가 모두 남고, 답은 가장 최근 값을 따릅니다.

In [9]:
# ▶ 제공 코드 — A-1에서 정정 후 다시 질문
ask("A-1", "정정할게. A라인 재가동은 15시로 변경됐어.")
ask("A-1", "A라인 재가동은 몇 시야?")
show_history("A-1")

[A-1] 사용자: 정정할게. A라인 재가동은 15시로 변경됐어.
[A-1] 답변  : 정정했습니다. A라인 재가동은 15시 예정입니다.

[A-1] 사용자: A라인 재가동은 몇 시야?
[A-1] 답변  : A라인 재가동은 15시 예정입니다.

--- [A-1] 저장된 메시지 8개 ---
 1. human: A라인은 14시에 재가동할 예정이야.
 2. ai   : A라인은 14시에 재가동 예정으로 기록했습니다.
 3. human: A라인 재가동은 몇 시야?
 4. ai   : A라인 재가동은 14시 예정입니다.
 5. human: 정정할게. A라인 재가동은 15시로 변경됐어.
 6. ai   : 정정했습니다. A라인 재가동은 15시 예정입니다.
 7. human: A라인 재가동은 몇 시야?
 8. ai   : A라인 재가동은 15시 예정입니다.


[HumanMessage(content='A라인은 14시에 재가동할 예정이야.', additional_kwargs={}, response_metadata={}, id='d9c3558e-cec7-42c5-98ed-068752c53178'),
 AIMessage(content='A라인은 14시에 재가동 예정으로 기록했습니다.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 36, 'prompt_tokens': 90, 'total_tokens': 126, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 13, 'rejected_prediction_tokens': 0, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': 0, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}, 'input_tokens': 90, 'output_tokens': 36, 'cached_prompt_tokens': 0, 'cache_write_tokens': 0}, 'model_provider': 'openai', 'model_name': 'gpt-6-luna', 'system_fingerprint': None, 'id': 'chatcmpl-EoN6YgrBwdzED3cdpmkXGub5CIkC0', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None, 'normalized_finish_reason': 'stop'}, id='lc_run--01a21681-9a76-7476-baa1-a92eaea12565-0

### 정리
| 확인한 것 | 근거 |
|---|---|
| 따로 보낸 호출은 앞 문장을 받지 못함 | 단계 2의 두 번째 답 |
| 같은 thread_id에는 새 메시지 하나만 보내도 기록이 이어짐 | 단계 5의 A-1 메시지 수 |
| 새 thread_id는 빈 기록에서 시작 | 단계 5의 B-1 메시지 수 |
| 정정은 기록을 지우지 않고, 답은 최신 값을 따름 | 단계 6의 A-1 기록 |

InMemorySaver는 메모리에만 보관합니다. 커널을 재시작하거나 `new_chat_agent()`로 새로 만들면 기록이 사라집니다.

**이어서 할 일** — 런박스 과제에서 A라인 대화방의 정정과 B라인 대화방의 별도 기록을 직접 시험합니다. 안내: `바이브코딩_정정과대화방/실습_정정과대화방_바이브코딩.html`